In [ ]:
!pip install tensorflow


In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
# ===============================
# 1. Load Processed Data
# ===============================
df = pd.read_csv("processed_diabetes.csv")

print("Dataset loaded:", df.shape)


Dataset loaded: (768, 18)


In [4]:
# ===============================
# 2. Define Features & Target
# ===============================
X = df.drop('Outcome', axis=1)
y = df['Outcome']

print("Feature columns:", X.columns.tolist())

# Save feature order for Streamlit
import joblib
joblib.dump(X.columns.tolist(), "features.pkl")

Feature columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'BMI_Category', 'Age_Group', 'Glucose_Level', 'BMI_Age', 'Glucose_BMI', 'Insulin_Glucose_Ratio', 'Pregnancy_Age_Ratio', 'Pregnancy_Risk', 'Insulin_log']


['features.pkl']

In [5]:
# ===============================
# 3. Train-Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
# ===============================
# 4.Compute Class Weights (IMPORTANT)
# ===============================
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute weights
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dictionary
class_weights = dict(enumerate(weights))

print("Class Weights:", class_weights)

Class Weights: {0: np.float64(0.7675), 1: np.float64(1.4345794392523366)}


In [7]:
# ===============================
# 5. Feature Scaling (IMPORTANT)
# ===============================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save scaler for Streamlit
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [8]:
# ===============================
# 6. Build Improved ANN Model
# ===============================
model = Sequential()

# Input Layer + Hidden Layers
model.add(Dense(32, activation='relu', input_dim=X_train.shape[1]))
model.add(Dropout(0.3))  # prevent overfitting

model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(8, activation='relu'))

# Output Layer
model.add(Dense(1, activation='sigmoid'))

# Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 32)                  │             576 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 16)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 8)                   │             136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │               9 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,249 (4.88 KB)

 Trainable params: 1,249 (4.88 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# ===============================
# 7. Training with Early Stopping + Class Weights
# ===============================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    class_weight=class_weights,
    verbose=1
)

Epoch 1/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5418 - loss: 0.6831 - val_accuracy: 0.6911 - val_loss: 0.6448
Epoch 2/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6273 - loss: 0.6331 - val_accuracy: 0.7561 - val_loss: 0.6075
Epoch 3/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6884 - loss: 0.6101 - val_accuracy: 0.7398 - val_loss: 0.5639
Epoch 4/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7006 - loss: 0.5835 - val_accuracy: 0.7398 - val_loss: 0.5369
Epoch 5/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7291 - loss: 0.5599 - val_accuracy: 0.7561 - val_loss: 0.5123
Epoch 6/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7393 - loss: 0.5630 - val_accuracy: 0.7642 - val_loss: 0.4947
Epoch 7/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7291 - loss: 0.5393 - val_accuracy: 0.7398 - val_loss: 0.4843
Epoch 8/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7576 - loss: 0.5235 - val_accuracy: 0.7561 - v

In [10]:
# ===============================
# 8. Model Evaluation
# ===============================
y_pred = (model.predict(X_test) > 0.5).astype(int)
y_prob = model.predict(X_test)

print("Sample probabilities:", y_prob[:10])
print("\nModel Evaluation:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Sample probabilities: [[0.7924147 ]
 [0.33968052]
 [0.3944677 ]
 [0.4237757 ]
 [0.07447121]
 [0.2461007 ]
 [0.66934144]
 [0.8905424 ]
 [0.09486401]
 [0.856655  ]]

Model Evaluation:
Accuracy: 0.7272727272727273

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.74      0.78       100
           1       0.59      0.70      0.64        54

    accuracy                           0.73       154
   macro avg       0.71      0.72      0.71       154
weighted avg       0.74      0.73      0.73       154


Confusion Matrix:
 [[74 26]
 [16 38]]


In [11]:
# ===============================
# 9. Save Model
# ===============================
model.save("diabetes_ann_model.h5")

print("\nModel and scaler saved successfully!")


Model and scaler saved successfully!
